# BoltaNote — training
Pakistani banknote reader for blind people: 7 notes × front/back + "no note" = 15 classes.

**Runtime → Run all.** About 15 minutes on an L4 GPU. Approve Google Drive access when asked.
Results are saved to *Google Drive → Deep Learning Projects → BoltaNote → outputs*, one model at a time
(re-running skips finished models).

Three models, same data, same augmentation:
1. `mobilenetv4_conv_medium` — MobileNetV4 (2024), ImageNet-pretrained
2. `mobilenetv3_large` — MobileNetV3-Large (2019), ImageNet-pretrained
3. `cnn_scratch` — a 5-block CNN from random weights (baseline)

In [1]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/Deep Learning Projects/BoltaNote/outputs'
os.makedirs(OUT, exist_ok=True)
print('saving to', OUT)

GPU 0: NVIDIA L4 (UUID: GPU-7164d101-4280-3934-7981-d32b59aac738)
Mounted at /content/drive
saving to /content/drive/MyDrive/Deep Learning Projects/BoltaNote/outputs


In [2]:
# code (the same files as the GitHub repo)
!rm -rf /content/gist /content/boltanote && git clone -q https://gist.github.com/Hassan-Ali-Mehdi-3024/a5ded9f3d3091e8c86106c8cbefde439.git /content/gist && python /content/gist/boltanote_bundle.py /content/boltanote
%cd /content/boltanote
!pip install -q "timm>=1.0.9" imagehash
!ls -R /content/boltanote | head -20

extracted to /content/boltanote
/content/boltanote
/content/boltanote:
boltanote
data
requirements.txt
scripts
train.py

/content/boltanote/boltanote:
augment.py
data.py
metrics.py
models.py

/content/boltanote/data:
splits.csv

/content/boltanote/scripts:
get_data.sh
make_splits.py


In [3]:
# public data: Kaggle currency photos (CC0), Imagenette "no note" photos, Roboflow external test set
!bash scripts/get_data.sh /content/data

rafay: 3891  imagenette: 13394  roboflow: 5587


## 1. MobileNetV4-Conv-Medium

In [4]:
!python train.py --data /content/data --model mobilenetv4_conv_medium --out "{OUT}" --skip-done 2>&1 | grep --line-buffered -v -i warn | tee -a "{OUT}/train_log.txt"

{'train': 3429, 'val': 735, 'test': 727, 'external': 63} | device: cuda NVIDIA L4
cached 4954 photos at 320px in 11s
ep  1/40  loss 4.4029  val value-acc 0.7156  22s  *best
ep  2/40  loss 1.9097  val value-acc 0.8408  4s  *best
ep  3/40  loss 1.8900  val value-acc 0.7633  4s
ep  4/40  loss 1.7059  val value-acc 0.9415  4s  *best
ep  5/40  loss 1.4300  val value-acc 0.9320  4s
ep  6/40  loss 1.6789  val value-acc 0.5034  4s
ep  7/40  loss 2.1586  val value-acc 0.7810  4s
ep  8/40  loss 1.7852  val value-acc 0.8857  4s
ep  9/40  loss 1.4472  val value-acc 0.9401  4s
ep 10/40  loss 1.2652  val value-acc 0.9728  4s  *best
ep 11/40  loss 1.1422  val value-acc 0.9741  4s  *best
ep 12/40  loss 1.1456  val value-acc 0.9796  4s  *best
ep 13/40  loss 1.0397  val value-acc 0.9741  4s
ep 14/40  loss 1.0000  val value-acc 0.9905  4s  *best
ep 15/40  loss 0.9421  val value-acc 0.9905  4s
ep 16/40  loss 0.9217  val value-acc 0.9878  4s
ep 17/40  loss 0.8717  val value-acc 0.9918  4s  *best
ep 18/40  

## 2. MobileNetV3-Large

In [5]:
!python train.py --data /content/data --model mobilenetv3_large --out "{OUT}" --skip-done 2>&1 | grep --line-buffered -v -i warn | tee -a "{OUT}/train_log.txt"

{'train': 3429, 'val': 735, 'test': 727, 'external': 63} | device: cuda NVIDIA L4
cached 4954 photos at 320px in 11s
ep  1/40  loss 2.6597  val value-acc 0.8680  17s  *best
ep  2/40  loss 1.1486  val value-acc 0.9810  3s  *best
ep  3/40  loss 0.9092  val value-acc 0.9946  3s  *best
ep  4/40  loss 0.8208  val value-acc 0.9986  3s  *best
ep  5/40  loss 0.7538  val value-acc 0.9973  3s
ep  6/40  loss 0.7472  val value-acc 0.9986  3s
ep  7/40  loss 0.7033  val value-acc 0.9986  3s
ep  8/40  loss 0.6933  val value-acc 0.9986  3s
ep  9/40  loss 0.6903  val value-acc 0.9986  3s
ep 10/40  loss 0.6738  val value-acc 0.9986  3s
ep 11/40  loss 0.6634  val value-acc 0.9986  3s
ep 12/40  loss 0.6463  val value-acc 1.0000  3s  *best
ep 13/40  loss 0.6432  val value-acc 1.0000  3s
ep 14/40  loss 0.6406  val value-acc 1.0000  3s
ep 15/40  loss 0.6370  val value-acc 1.0000  3s
ep 16/40  loss 0.6310  val value-acc 1.0000  3s
ep 17/40  loss 0.6288  val value-acc 1.0000  3s
ep 18/40  loss 0.6298  val valu

## 3. CNN from scratch (baseline)

In [6]:
!python train.py --data /content/data --model cnn_scratch --out "{OUT}" --skip-done 2>&1 | grep --line-buffered -v -i warn | tee -a "{OUT}/train_log.txt"

{'train': 3429, 'val': 735, 'test': 727, 'external': 63} | device: cuda NVIDIA L4
cached 4954 photos at 320px in 11s
ep  1/80  loss 2.5236  val value-acc 0.3605  6s  *best
ep  2/80  loss 2.3350  val value-acc 0.3565  5s
ep  3/80  loss 2.2606  val value-acc 0.4707  5s  *best
ep  4/80  loss 2.1937  val value-acc 0.2803  5s
ep  5/80  loss 2.0895  val value-acc 0.6367  5s  *best
ep  6/80  loss 1.9137  val value-acc 0.7170  5s  *best
ep  7/80  loss 1.8484  val value-acc 0.5769  5s
ep  8/80  loss 1.7469  val value-acc 0.7429  5s  *best
ep  9/80  loss 1.6781  val value-acc 0.7864  5s  *best
ep 10/80  loss 1.5953  val value-acc 0.8299  5s  *best
ep 11/80  loss 1.5393  val value-acc 0.7211  5s
ep 12/80  loss 1.5011  val value-acc 0.8803  5s  *best
ep 13/80  loss 1.4476  val value-acc 0.8612  5s
ep 14/80  loss 1.4065  val value-acc 0.7075  5s
ep 15/80  loss 1.3722  val value-acc 0.8259  5s
ep 16/80  loss 1.3249  val value-acc 0.9306  5s  *best
ep 17/80  loss 1.2834  val value-acc 0.9293  5s
ep 1

## Results

In [7]:
import json, glob
print(f"{'model':26s} {'test acc':>8s} {'wrong':>6s} {'cover':>6s} {'external':>8s}")
for p in sorted(glob.glob(f'{OUT}/*/metrics.json')):
    m = json.load(open(p)); t = m['test']; e = m.get('external', {})
    print(f"{m['model']:26s} {t['value_accuracy']:8.4f} {t['speaking']['wrong_value_rate']:6.4f} {t['speaking']['coverage_on_notes']:6.3f} {e.get('value_accuracy', float('nan')):8.4f}")

model                      test acc  wrong  cover external
cnn_scratch                  0.9959 0.0028  0.998   0.8254
mobilenetv3_large            1.0000 0.0000  0.998   0.9206
mobilenetv4_conv_medium      0.9986 0.0014  1.000   0.9048


In [8]:
!cd "{OUT}/.." && rm -f boltanote_outputs.zip && zip -qr boltanote_outputs.zip outputs && ls -la "{OUT}/.."
print('DONE')

total 55187
drwx------ 4 root root     4096 Oct 27 21:30 .
drwx------ 4 root root     4096 Oct 27 20:56 ..
-rw------- 1 root root 56479419 Oct 27 21:30 boltanote_outputs.zip
-rw------- 1 root root    15325 Oct 27 21:30 BoltaNote_train.ipynb
drwx------ 2 root root     4096 Oct 27 21:08 my-photos
drwx------ 5 root root     4096 Oct 27 21:30 outputs
DONE
